# 生成图

In [2]:
import torch
import torch.nn.functional as F
import glob
import circuitgraph as cg
import networkx as nx
from torch_geometric.data import Data, HeteroData
from torch_geometric.utils.convert import from_networkx

## 1 转化

提取一下硬件类型和硬件名

In [4]:
DATASET_DIR = os.path.join('..', 'dataset', 'Trust-Hub_Hardware Obfuscation Modified')
filelist = glob.glob(os.path.join(DATASET_DIR, '**', '*.v'), recursive=True)
for file in filelist:
    # print(file.split('\\'))
    dir_list = file.split(os.sep)
    name = dir_list[-2]
    type = dir_list[-3]
    print(type, name)

c1355 c1355
c1355 c1355-CS1280
c1355 c1355-CS2560
c1355 c1355-CS320
c1355 c1355-CS640
c1355 c1355-CY1950
c1355 c1355-CY260
c1355 c1355-CY3720
c1355 c1355-CY460
c1355 c1355-CY980
c1355 c1355-NC1550
c1355 c1355-NC1870
c1355 c1355-NC2510
c1355 c1355-NC3790
c1355 c1355-NR1550
c1355 c1355-NR1870
c1355 c1355-NR2510
c1355 c1355-NR3790
c1355 c1355-NS1550
c1355 c1355-NS1870
c1355 c1355-NS2510
c1355 c1355-NS3790
c1355 c1355-RN1280
c1355 c1355-RN2560
c1355 c1355-RN320
c1355 c1355-RN640
c1355 c1355-SL1280
c1355 c1355-SL2560
c1355 c1355-SL320
c1355 c1355-SL640
c1908 c1908
c1908 c1908-BE320
c1908 c1908-BR320
c1908 c1908-BS320
c1908 c1908-CS1280
c1908 c1908-CS2560
c1908 c1908-CS320
c1908 c1908-CS640
c1908 c1908-CY2010
c1908 c1908-CY270
c1908 c1908-CY3830
c1908 c1908-CY480
c1908 c1908-CY990
c1908 c1908-NC1310
c1908 c1908-NC1630
c1908 c1908-NC2270
c1908 c1908-NC3550
c1908 c1908-NR1310
c1908 c1908-NR1630
c1908 c1908-NR2270
c1908 c1908-NR3550
c1908 c1908-NS1310
c1908 c1908-NS1630
c1908 c1908-NS2270
c1908

---

In [2]:
# convert type to label
type_label = {
    'c1355': 0,
    'c1908': 1,
    'c2670': 2,
    'c3540': 3,
    'c432': 4,
    'c499': 0,
    'c5315': 5,
    'c6288': 6,
    'c7552': 7,
    'c880': 8}

In [7]:
node_type_names = [
    'input',
    'and',
    'nand',
    'or',
    'nor',
    'not',
    'buf',
    'xor',
    'xnor',
    'output'
]
node_type_names_index = {}  # convert node type to index / one-hot
for index, node_type in enumerate(node_type_names):
    node_type_names_index[node_type] = index

In [3]:
# convert node type to index / one-hot
node_type_names_index = {
    'input': 0,
    'and': 1,
    'nand': 2,
    'or': 3,
    'nor': 4,
    'not': 5,
    'buf': 6,
    'xor': 7,
    'xnor': 8,
    'output': 9}

In [5]:
data_list = []  # store all data in a list

DATASET_DIR = os.path.join('..', 'dataset', 'Trust-Hub_Hardware Obfuscation Modified')
filelist = glob.glob(os.path.join(DATASET_DIR, '**', '*.v'), recursive=True)

for index, file in enumerate(filelist):
    dir_list = file.split(os.sep)
    hw_name = dir_list[-2]  # name attribute
    hw_type = dir_list[-3]  # type attribute

    # try:
    #     c = cg.from_file(file)  # circuitgraph
    # except Exception as e:
    #     print(repr(e), index, file)

    c = cg.from_file(file)  # circuitgraph

    c_modi = nx.DiGraph(c.graph)
    for node in c.graph.nodes:  # add output node
        try:
            is_output = c_modi.nodes[node]['output']
        except KeyError:
            is_output = False

        del c_modi.nodes[node]['output']  # delete output attribute

        if is_output:
            node_name = f"{node}_O"
            c_modi.add_node(node_name, type='output')
            c_modi.add_edge(node, node_name)

    data = from_networkx(c_modi)  # NetworkX to PyG
    
    node_type = []
    for type in data.type:
        node_type.append(node_type_names_index[type])

    data.x = torch.zeros(data.num_nodes, len(node_type_names_index))
    for index, type in enumerate(data.type):
        data.x[index, node_type[index]] = 1.0  # one-hot feature

    del data.num_nodes
    del data.type

    data.label = type_label[hw_type]  # add label
    data.type = hw_type  # add type attribute
    data.name = hw_name  # add name attribute

    data_list.append(data)
    print(data)

Data(edge_index=[2, 1096], x=[619, 10], label=0, type='c1355', name='c1355')
Data(edge_index=[2, 1419], x=[942, 10], label=0, type='c1355', name='c1355-CS1280')
Data(edge_index=[2, 1742], x=[1265, 10], label=0, type='c1355', name='c1355-CS2560')
Data(edge_index=[2, 1178], x=[701, 10], label=0, type='c1355', name='c1355-CS320')
Data(edge_index=[2, 1261], x=[784, 10], label=0, type='c1355', name='c1355-CS640')
Data(edge_index=[2, 2461], x=[1594, 10], label=0, type='c1355', name='c1355-CY1950')
Data(edge_index=[2, 1278], x=[749, 10], label=0, type='c1355', name='c1355-CY260')
Data(edge_index=[2, 3700], x=[2479, 10], label=0, type='c1355', name='c1355-CY3720')
Data(edge_index=[2, 1418], x=[849, 10], label=0, type='c1355', name='c1355-CY460')
Data(edge_index=[2, 1782], x=[1109, 10], label=0, type='c1355', name='c1355-CY980')
Data(edge_index=[2, 1528], x=[969, 10], label=0, type='c1355', name='c1355-NC1550')
Data(edge_index=[2, 1617], x=[1058, 10], label=0, type='c1355', name='c1355-NC1870')

In [6]:
torch.save(data_list, "NetlistGraph.pt")

---

搞一个带节点类型的版本，备用

In [3]:
data_list = torch.load("NetlistGraph.pt")

In [13]:
data_list[0].x.shape

torch.Size([619, 10])

In [12]:
for index, i in enumerate(data_list[0].x):
    print(index, i)

0 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
1 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
2 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
3 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
4 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
5 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
6 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
7 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
8 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
9 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
10 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
11 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
12 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
13 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
14 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
15 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
16 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
17 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
18 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
19 tensor([1., 0., 0.,

In [8]:
# data_list[0].x.argmax()
_, index = torch.max(data_list[0].x, dim=1)

In [11]:
len(index)

619

In [14]:
for data in data_list:
    _, node_type = torch.max(data.x, dim=1)
    data.node_type = node_type

In [15]:
data_list

[Data(edge_index=[2, 1096], x=[619, 10], label=0, type='c1355', name='c1355', node_type=[619]),
 Data(edge_index=[2, 1419], x=[942, 10], label=0, type='c1355', name='c1355-CS1280', node_type=[942]),
 Data(edge_index=[2, 1742], x=[1265, 10], label=0, type='c1355', name='c1355-CS2560', node_type=[1265]),
 Data(edge_index=[2, 1178], x=[701, 10], label=0, type='c1355', name='c1355-CS320', node_type=[701]),
 Data(edge_index=[2, 1261], x=[784, 10], label=0, type='c1355', name='c1355-CS640', node_type=[784]),
 Data(edge_index=[2, 2461], x=[1594, 10], label=0, type='c1355', name='c1355-CY1950', node_type=[1594]),
 Data(edge_index=[2, 1278], x=[749, 10], label=0, type='c1355', name='c1355-CY260', node_type=[749]),
 Data(edge_index=[2, 3700], x=[2479, 10], label=0, type='c1355', name='c1355-CY3720', node_type=[2479]),
 Data(edge_index=[2, 1418], x=[849, 10], label=0, type='c1355', name='c1355-CY460', node_type=[849]),
 Data(edge_index=[2, 1782], x=[1109, 10], label=0, type='c1355', name='c1355-C

In [16]:
torch.save(data_list, "NetlistGraph-node_type.pt")

---

In [22]:
len(data_list)

305

In [20]:
data_list[0]

Data(edge_index=[2, 1096], x=[619, 10], label=0, type='c1355', name='c1355')

In [21]:
data_list[0].x

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 0., 0., 1.]])